# Data Exploration & Cleaning Notes

This notebook was used during development to:
- explore the dataset structure
- identify data quality issues
- test cleaning rules (e.g. handling invalid values)
- validate results before implementing the final program

The final solution is implemented in the Python CLI application.

In [126]:
import pandas as pd
import numpy as np

# Import Data Set

In [127]:
# Import csv file as a DataFrame object:
df = pd.read_csv("./worldData.csv")

In [128]:
# Visualise: 
df

,Unnamed: 0,iso_a2,iso_a2.1,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
0,1,FJ,FJ,Fiji,Oceania,Oceania,Melanesia,Sovereign country,1.928997e+04,885806.0,69.960000,8222.253784
1,1,FJ,FJ,Fiji,Oceania,Oceania,Melanesia,Sovereign country,1.928997e+04,885806.0,69.960000,8222.253784
2,2,TZ,TZ,Tanzania,Africa,Africa,Eastern Africa,Sovereign country,9.327458e+05,52234869.0,64.163000,2402.099404
3,3,EH,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
4,4,CA,CA,Canada,North America,Americas,Northern America,Sovereign country,1.003604e+07,35535348.0,81.953049,43079.142520
...,...,...,...,...,...,...,...,...,...,...,...,...
184,174,ME,ME,Montenegro,Europe,Europe,Southern Europe,Sovereign country,1.344368e+04,621810.0,76.712000,14796.635400
185,175,XK,XK,Kosovo,Europe,Europe,Southern Europe,Sovereign country,1.123026e+04,1821800.0,71.097561,8698.291559
186,175,XK,XK,Kosovo,Europe,Europe,Southern Europe,Sovereign country,1.123026e+04,1821800.0,71.097561,8698.291559
187,176,TT,TT,Trinidad and Tobago,North America,Americas,Caribbean,Sovereign country,7.737810e+03,1354493.0,70.426000,31181.821200


# Data Cleaning 

## Duplicate Columns (iso_a2)
The first problem is a duplicate column, "iso_a2".

First, check if these columns are actually the same:

In [129]:
# Return all rows where column1 and column2 do not match:
df[df.iso_a2 != (df["iso_a2.1"])]

,Unnamed: 0,iso_a2,iso_a2.1,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
54,51,NaN,NaN,Namibia,Africa,Africa,Southern Africa,Sovereign country,824886.561000,2370992.0,62.981,9617.396957
170,161,NaN,NaN,Northern Cyprus,Asia,Asia,Western Asia,Sovereign country,3786.364506,NaN,NaN,NaN
178,168,NaN,NaN,Somaliland,Africa,Africa,Eastern Africa,Indeterminate,167349.612700,NaN,NaN,NaN


The 'mismatched' rows are only due to Null cells, so yes they are the same.

Therefore one can be removed:

In [130]:
# Remove duplicate column by altering the existing DataFrame: 
df.drop(columns=["iso_a2.1"], inplace=True, axis=1)
df

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
0,1,FJ,Fiji,Oceania,Oceania,Melanesia,Sovereign country,1.928997e+04,885806.0,69.960000,8222.253784
1,1,FJ,Fiji,Oceania,Oceania,Melanesia,Sovereign country,1.928997e+04,885806.0,69.960000,8222.253784
2,2,TZ,Tanzania,Africa,Africa,Eastern Africa,Sovereign country,9.327458e+05,52234869.0,64.163000,2402.099404
3,3,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
4,4,CA,Canada,North America,Americas,Northern America,Sovereign country,1.003604e+07,35535348.0,81.953049,43079.142520
...,...,...,...,...,...,...,...,...,...,...,...
184,174,ME,Montenegro,Europe,Europe,Southern Europe,Sovereign country,1.344368e+04,621810.0,76.712000,14796.635400
185,175,XK,Kosovo,Europe,Europe,Southern Europe,Sovereign country,1.123026e+04,1821800.0,71.097561,8698.291559
186,175,XK,Kosovo,Europe,Europe,Southern Europe,Sovereign country,1.123026e+04,1821800.0,71.097561,8698.291559
187,176,TT,Trinidad and Tobago,North America,Americas,Caribbean,Sovereign country,7.737810e+03,1354493.0,70.426000,31181.821200


## De-Duplication

### Remove Exact Duplicates:

Now to remove duplicate rows. 

Ideally this dataset needs one entry per country. 

First, I can safely remove rows that are exact duplicates of another:

In [131]:
# Remove any rows that are exact duplicates of another row: 
clean_df = df.drop_duplicates()

### Identify Other Duplicates:

Now I will check for duplicates of the same country that have differing information.

First, I will print out the number of rows after removing duplicates in different columns, to gauge how many duplicates exist in the dataset:

In [132]:
# Identify non-exact duplicates:
nm_clean_df = clean_df.drop_duplicates(subset=["name_long"])
ch1_clean_df = clean_df.drop_duplicates(subset=["iso_a2"])
unamed_clean_df = clean_df.drop_duplicates(subset=["Unnamed: 0"])

print(f"Original Rows Size: {df.shape[0]}")
print(f"Unnamed Dupes Rows Size: {unamed_clean_df.shape[0]}")
print(f"Exact Dupes Removed Rows: {clean_df.shape[0]}")
print(f"Long Name Dupes Removed Rows: {nm_clean_df.shape[0]}")
print(f"Short Name Dupes Removed Rows: {ch1_clean_df.shape[0]}")

Original Rows Size: 189
Unnamed Dupes Rows Size: 177
Exact Dupes Removed Rows: 179
Long Name Dupes Removed Rows: 177
Short Name Dupes Removed Rows: 175


In [133]:
# Analyse name_long duplicates for inconsistencies:
bool_mask = clean_df.duplicated(subset=["name_long"], keep=False)
clean_df[bool_mask]

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
55,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.4389,14546111.0,66.376,2218.551917
56,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.4389,14546111.0,0.000,2218.551917
87,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.3216,11143908.0,75.335,10767.027650
88,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.3216,11143908.0,75.335,0.000000


In [134]:
# Analyse iso_a2 duplicates for inconsistencies:
bool_mask = clean_df.duplicated(subset=["iso_a2"], keep=False)
clean_df[bool_mask]

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
54,51,NaN,Namibia,Africa,Africa,Southern Africa,Sovereign country,824886.561000,2370992.0,62.981,9617.396957
55,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.438900,14546111.0,66.376,2218.551917
56,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.438900,14546111.0,0.000,2218.551917
87,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.321600,11143908.0,75.335,10767.027650
88,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.321600,11143908.0,75.335,0.000000
170,161,NaN,Northern Cyprus,Asia,Asia,Western Asia,Sovereign country,3786.364506,NaN,NaN,NaN
178,168,NaN,Somaliland,Africa,Africa,Eastern Africa,Indeterminate,167349.612700,NaN,NaN,NaN


I can see that some of these 'duplicates' are actually different countries that have missing iso_a2 codes. Since these are not actual duplicates of countries and contain other information useful for filtering and insights, I will leave them in the DataFrame.

The 'true' duplicates (Tunisia & Senegal) have only not been removed thus far due to missing values which their counterparts contain. Therefore, these can be removed safely.

### Remove Duplicate Rows With Missing Data

##### Pre Removal Check:

In [135]:
# Visualise the duplicates: 
clean_df[(clean_df.name_long == "Senegal") | (clean_df.name_long == "Tunisia")]

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
55,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.4389,14546111.0,66.376,2218.551917
56,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.4389,14546111.0,0.000,2218.551917
87,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.3216,11143908.0,75.335,10767.027650
88,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.3216,11143908.0,75.335,0.000000


##### Remove Duplicates:

In [136]:
# Drop rows that are duplicates and have lifeExp < 1.0 or gdpPercap < 1.0:
copy_df = clean_df.drop(clean_df[(bool_mask) & ((clean_df.lifeExp < 1.0) | (clean_df.gdpPercap < 1.0))].index) 

##### Post-Removal Check:

In [137]:
# Only rows 55 and 87 should still exist:
copy_df[(copy_df.name_long == "Senegal") | (copy_df.name_long == "Tunisia")]

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
55,52,SN,Senegal,Africa,Africa,Western Africa,Sovereign country,194390.4389,14546111.0,66.376,2218.551917
87,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,156237.3216,11143908.0,75.335,10767.027650


##### Double Check Full Dataset Before Saving to clean_df:

In [138]:
copy_df

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
0,1,FJ,Fiji,Oceania,Oceania,Melanesia,Sovereign country,1.928997e+04,885806.0,69.960000,8222.253784
2,2,TZ,Tanzania,Africa,Africa,Eastern Africa,Sovereign country,9.327458e+05,52234869.0,64.163000,2402.099404
3,3,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
4,4,CA,Canada,North America,Americas,Northern America,Sovereign country,1.003604e+07,35535348.0,81.953049,43079.142520
5,5,US,United States,North America,Americas,Northern America,Country,9.510744e+06,318622525.0,78.841463,51921.984640
...,...,...,...,...,...,...,...,...,...,...,...
183,173,RS,Serbia,Europe,Europe,Southern Europe,Sovereign country,7.638861e+04,0.0,75.336585,13112.908960
184,174,ME,Montenegro,Europe,Europe,Southern Europe,Sovereign country,1.344368e+04,621810.0,76.712000,14796.635400
185,175,XK,Kosovo,Europe,Europe,Southern Europe,Sovereign country,1.123026e+04,1821800.0,71.097561,8698.291559
187,176,TT,Trinidad and Tobago,North America,Americas,Caribbean,Sovereign country,7.737810e+03,1354493.0,70.426000,31181.821200


In [139]:
clean_df = copy_df

## Rebuild Missing Data


Now I need to see if there are any cells with missing data. 

First, I will visualise these cells for analysis:
    

In [140]:
# Create mask to filter rows with missing data:
missing_vals_mask = (
    (df["iso_a2"].isna()) |
    (df["area_km2"].isna()) |
    (df["pop"].isna()) |
    (df["pop"] < 1) |
    (df["lifeExp"].isna()) |
    (df["lifeExp"] < 1) |
    (df["gdpPercap"].isna()) |
    (df["gdpPercap"] < 1.0) |
    (df["lifeExp"] > 100)
)

# Apply mask and return filtered results:
missing_df = df[missing_vals_mask]

# Visualise rows:
missing_df

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
3,3,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
7,7,UZ,Uzbekistan,Asia,Asia,Central Asia,Sovereign country,4.614103e+05,30757700.0,71.039000,-5370.865802
13,13,SO,Somalia,Africa,Africa,Eastern Africa,Sovereign country,4.843328e+05,13513125.0,55.467000,NaN
22,21,FK,Falkland Islands,South America,Americas,South America,Dependency,1.636380e+04,NaN,NaN,NaN
23,22,NO,Norway,Europe,Europe,Northern Europe,Sovereign country,3.979946e+05,NaN,NaN,NaN
24,23,GL,Greenland,North America,Americas,Northern America,Country,2.206644e+06,56295.0,NaN,NaN
25,24,TF,French Southern and Antarctic Lands,Seven seas (open ocean),Seven seas (open ocean),Seven seas (open ocean),Dependency,1.160257e+04,NaN,NaN,NaN
27,26,ZA,South Africa,Africa,Africa,Southern Africa,Sovereign country,1.216401e+06,54539571.0,609.930000,12389.714670
46,44,FR,France,Europe,Europe,Western Europe,Country,6.448479e+05,NaN,NaN,-5.000000
51,48,CU,Cuba,North America,Americas,Caribbean,Sovereign country,1.148662e+05,11439767.0,79.415000,NaN


#### Insights from the missing data above:
1. There is a mix of NaN and Zero values. Standardizing them to a single format would make future querying and filtering easier to implement. 
2. Missing data only exists in **iso_a2**, **pop**, **lifeExp**, and **gdpPercap** columns.
3. While some information in the **type** column is set to **"indeterminate"**, this seems to be due to the row representing an ambiguous area of land rather than a true country, so I will not treat this as missing data.
4. Some float and integer type cells contain **negative values**. Due to the nature of what these cells represent, this is most likely an error and they can be treated as missing data.

#### Cleaning Approach:
1. While some entries, such as *France* contain a lot of missing data, every row seems to at least have accurate information regarding surface area and regional definitions. This means the rows can still be used to gather insights related to these fields. Therefore, I will not delete entire rows.
2. Since NaN, zero-value, and negative-value cells all represent the same level of missing data, I will standardize them to a single format. All missing data cells will be standardized to NaN.
3. NaN cells in the **iso_a2** will remain as such, as I don't want to directly tamper with the data.

 (PS) I later discovered South Africa has an extreme lifeExpectancy value of 609. Therefore I will add a rule to change any values over 100 in this column to NaN. 

In [149]:
def clean_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    
    # Define the columns to be edited: 
    cols = ["pop", "lifeExp", "gdpPercap"]
    
    # Any data that is less than or equal to Zero, set to NaN (this is the default replacement of the DataFrame.mask() function.)
    df[cols] = df[cols].mask(df[cols] <= 0)

    # Any lifeExp value that is over 100, set to NaN
    df["lifeExp"] = df["lifeExp"].mask(df["lifeExp"] > 100)

    return df


In [150]:

# Create copy to test removal before applying to the actual dataset:
filled_df = clean_dataframe(missing_df.copy())


# Visualise result:
filled_df

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
3,3,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
7,7,UZ,Uzbekistan,Asia,Asia,Central Asia,Sovereign country,4.614103e+05,30757700.0,71.039000,NaN
13,13,SO,Somalia,Africa,Africa,Eastern Africa,Sovereign country,4.843328e+05,13513125.0,55.467000,NaN
22,21,FK,Falkland Islands,South America,Americas,South America,Dependency,1.636380e+04,NaN,NaN,NaN
23,22,NO,Norway,Europe,Europe,Northern Europe,Sovereign country,3.979946e+05,NaN,NaN,NaN
24,23,GL,Greenland,North America,Americas,Northern America,Country,2.206644e+06,56295.0,NaN,NaN
25,24,TF,French Southern and Antarctic Lands,Seven seas (open ocean),Seven seas (open ocean),Seven seas (open ocean),Dependency,1.160257e+04,NaN,NaN,NaN
27,26,ZA,South Africa,Africa,Africa,Southern Africa,Sovereign country,1.216401e+06,54539571.0,NaN,12389.714670
46,44,FR,France,Europe,Europe,Western Europe,Country,6.448479e+05,NaN,NaN,NaN
51,48,CU,Cuba,North America,Americas,Caribbean,Sovereign country,1.148662e+05,11439767.0,79.415000,NaN


Everything looks in order, so I will now apply this standarization to the actual dataset:

In [151]:
# Replace messy columns with standardised columns:
clean_df = clean_dataframe(clean_df.copy())

# Visualize result:
clean_df

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
0,1,FJ,Fiji,Oceania,Oceania,Melanesia,Sovereign country,1.928997e+04,885806.0,69.960000,8222.253784
2,2,TZ,Tanzania,Africa,Africa,Eastern Africa,Sovereign country,9.327458e+05,52234869.0,64.163000,2402.099404
3,3,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
4,4,CA,Canada,North America,Americas,Northern America,Sovereign country,1.003604e+07,35535348.0,81.953049,43079.142520
5,5,US,United States,North America,Americas,Northern America,Country,9.510744e+06,318622525.0,78.841463,51921.984640
...,...,...,...,...,...,...,...,...,...,...,...
183,173,RS,Serbia,Europe,Europe,Southern Europe,Sovereign country,7.638861e+04,NaN,75.336585,13112.908960
184,174,ME,Montenegro,Europe,Europe,Southern Europe,Sovereign country,1.344368e+04,621810.0,76.712000,14796.635400
185,175,XK,Kosovo,Europe,Europe,Southern Europe,Sovereign country,1.123026e+04,1821800.0,71.097561,8698.291559
187,176,TT,Trinidad and Tobago,North America,Americas,Caribbean,Sovereign country,7.737810e+03,1354493.0,70.426000,31181.821200


# Answering Question About Data

## Which continent has the most countries in the data?

If data is clean and each row represents one country, this is essentially "which appears most frequently in the 'continent' column?"

In [165]:
# Get the count of every continent:
continent_counts = clean_df["continent"].value_counts()
print(continent_counts)

continent
Africa                     51
Asia                       47
Europe                     39
North America              18
South America              13
Oceania                     7
Seven seas (open ocean)     1
Antarctica                  1
Name: count, dtype: int64


In [167]:
max_continent_value = continent_counts.max()
max_continent_name = continent_counts.idxmax()
print(f"{max_continent_name} {max_continent_value}")

Africa 51


## Which region has the largest combined area in sq. km?

In [152]:
# Group regions by their combined total area_km2:
region_areas = clean_df.groupby("region_un")["area_km2"].sum()

#Visualise: 
print(region_areas)



region_un
Africa                     2.994620e+07
Americas                   4.224690e+07
Antarctica                 1.233596e+07
Asia                       3.125246e+07
Europe                     2.306522e+07
Oceania                    8.504489e+06
Seven seas (open ocean)    1.160257e+04
Name: area_km2, dtype: float64


In [153]:
# Return region with the highest km2 value:
largest_region_name = region_areas.idxmax()
largest_region_val = region_areas.max()

print(f"{largest_region_name}\n{largest_region_val}")

Americas
42246901.542602


Interestingly, Asia does not return as the largest region. This could be due to missing or filtered data in the dataset after cleaning.

## Which country has the highest life expectancy?

In [154]:
# Visualise data:
world_lifeExp = clean_df.groupby("name_long")["lifeExp"].sum()
print(f"Median life expectancy: {world_lifeExp.median()}\n")
print(f"Top Ten Highest Life Expectancies:\n{world_lifeExp.nlargest(10)}\n")
print(f"Top Ten Lowest Life Expectancies:\n{world_lifeExp.nsmallest(10)}")



Median life expectancy: 71.62

Top Ten Highest Life Expectancies:
name_long
Japan          83.587805
Spain          83.229268
Switzerland    83.197561
Italy          83.090244
Iceland        82.860976
Australia      82.300000
Sweden         82.253659
Luxembourg     82.229268
Israel         82.153659
Canada         81.953049
Name: lifeExp, dtype: float64

Top Ten Lowest Life Expectancies:
name_long
Antarctica                             0.0
Falkland Islands                       0.0
France                                 0.0
French Southern and Antarctic Lands    0.0
Greenland                              0.0
Northern Cyprus                        0.0
Norway                                 0.0
Philippines                            0.0
Somaliland                             0.0
South Africa                           0.0
Name: lifeExp, dtype: float64


In [170]:
# Return row with the highest life expentancy value:
name = world_lifeExp.idxmax()
value = world_lifeExp.max()
print(f"{name} {value}")


Japan 83.58780488


New Result:
Japan at 83 years.

Original Result:
Unexpectedly, South Africa has an average life expectancy of 609 years. Clearly this is a mistake.

I'm guessing that the culprit is a misplaced decimal point rather than corrupted data, as every other column's data seems reasonable.
South Africa's true life expectancy is probably 60.9 years.

However, this is still just a guess. So the safest option to fix this is to alter my cleaning algorithm to revert extreme lifeExp values to NaN.

## Which subregion has the lowest / highest average GDP per capita?

In [171]:
# Group subregions by their average(mean) gdpPercap:
subregion_gdp = clean_df.groupby("subregion")["gdpPercap"].mean()

In [172]:
# Visualise results:
print(subregion_gdp.sort_values(ascending=False))

subregion
Western Europe               56884.584962
Northern America             47500.563580
Australia and New Zealand    39001.264350
Northern Europe              36058.752415
Western Asia                 31791.457802
Eastern Asia                 20481.537513
Eastern Europe               19712.829171
Caribbean                    19511.970865
Southern Europe              19510.242812
South-Eastern Asia           17051.592972
South America                13761.997210
Central Asia                 10911.961484
Central America              10258.359233
Southern Africa               9693.990137
Northern Africa               9079.476234
Middle Africa                 8333.981095
Southern Asia                 6497.571240
Melanesia                     4240.809443
Western Africa                2221.161155
Eastern Africa                1796.434045
Antarctica                            NaN
Seven seas (open ocean)               NaN
Name: gdpPercap, dtype: float64


In [158]:
subregion_gdp.dropna(inplace=True)

max_gdp_value = subregion_gdp.max()
max_gdp_name = subregion_gdp.idxmax()

min_gdp_value = subregion_gdp.min()
min_gdp_name = subregion_gdp.idxmin()

print(f"Highest GDP:\n{max_gdp_name}\n{max_gdp_value} average GDP per capita\n")
print(f"Lowest GDP:\n{min_gdp_name}\n{min_gdp_value} average GDP per capita")

Highest GDP:
Western Europe
56884.584962 average GDP per capita

Lowest GDP:
Eastern Africa
1796.4340450363636 average GDP per capita


# Filter and Summary

Allow filtering for one column at a time, implement combination filters later.

In [203]:
def filter_data(
    df: pd.DataFrame,
    continent: str | None = None,
    region: str | None = None,
    subregion: str | None = None,
    country_type: str | None = None
) -> pd.DataFrame:
    filtered_df = df.copy()
    #Combination filter, only works if all parameters are valid:
    # return filtered_df[(filtered_df.continent == continent) & 
    #         (filtered_df.region_un == region) &
    #         (filtered_df.subregion == subregion) &
    #          (filtered_df.type == country_type)]
    if continent:
        filtered_df = filtered_df[filtered_df["continent"].str.lower() == continent.lower()]

    if region:
        filtered_df = filtered_df[filtered_df["region_un"].str.lower() == region.lower()]

    if subregion:
        filtered_df = filtered_df[filtered_df["subregion"].str.lower() == subregion.lower()]

    if country_type:
        filtered_df = filtered_df[filtered_df["type"].str.lower() == country_type.lower()]

    return filtered_df

    

In [208]:
# Test filter function
input_continent = "Africa"
input_region = "Africa"
subregion = "Northern Africa"
country_type = "Sovereign country"

filtered_df = filter_data(df=clean_df, subregion=subregion)

filtered_df.drop(columns="Unnamed: 0", inplace=True)

filtered_df.describe()


,area_km2,pop,lifeExp,gdpPercap
count,7.000000e+00,6.000000e+00,6.000000,5.000000
mean,1.091580e+06,3.672165e+07,72.177667,9079.476234
std,8.654388e+05,3.044037e+07,4.472089,3563.986105
min,9.627060e+04,6.204108e+06,64.002000,4188.334814
25%,3.739782e+05,1.693745e+07,71.254750,7078.881489
50%,9.963116e+05,3.602800e+07,73.484000,9879.799359
75%,1.742303e+06,3.876946e+07,75.328500,10767.027650
max,2.315917e+06,9.181257e+07,75.641000,13483.337860


In [207]:
filtered_df

,Unnamed: 0,iso_a2,name_long,continent,region_un,subregion,type,area_km2,pop,lifeExp,gdpPercap
3,3,EH,Western Sahara,Africa,Africa,Northern Africa,Indeterminate,9.627060e+04,NaN,NaN,NaN
15,15,SD,Sudan,Africa,Africa,Northern Africa,Sovereign country,1.850886e+06,37737913.0,64.002,4188.334814
87,82,TN,Tunisia,Africa,Africa,Northern Africa,Sovereign country,1.562373e+05,11143908.0,75.335,10767.027650
89,83,DZ,Algeria,Africa,Africa,Northern Africa,Sovereign country,2.315917e+06,39113313.0,75.641,13483.337860
172,163,MA,Morocco,Africa,Africa,Northern Africa,Sovereign country,5.917190e+05,34318082.0,75.309,7078.881489
173,164,EG,Egypt,Africa,Africa,Northern Africa,Sovereign country,9.963116e+05,91812566.0,71.120,9879.799359
174,165,LY,Libya,Africa,Africa,Northern Africa,Sovereign country,1.633721e+06,6204108.0,71.659,NaN
